# v005 results and live progress

Read-only review of completed A sequence/tree records, unfinished stages, and locked future confirmation manifests. This notebook never trains, changes models/weights, tunes ensemble choices, reads unlabeled test inputs, or submits. It discovers model counts dynamically; an interrupted stage is **in progress**, even if it has a promising best-so-far score.

Learning curves compare native-unit A inspiratory MAE with a fixed **evaluation-mode fitting sample**. Optional online training MAE uses training-mode predictions while weights change, so it is a separate diagnostic. The A ranking is exploratory selection evidence. The accepted v004b Gym MAE is .47462; its earlier validation blend MAE is .483719 on A. Final all-data models have no independent A/C holdout.

Defaults verify available artifacts and save review tables/figures under `outputs/improvements_v005/review` once v005 results exist. Set `PA3_REVIEW_V005_VERIFY=0` to skip expensive file hashing, `PA3_REVIEW_V005_SAVE=0` for a display-only view, or `PA3_REVIEW_V005_ONLINE=1` to add the explicitly labeled online curve. With no v005 results, execution needs only the Python standard library and writes nothing. Rerun to refresh live progress; this notebook does not poll or contact remote services.


In [1]:
import os, csv, hashlib, json, math, time, ast
from pathlib import Path
from datetime import datetime, timezone
PROJECT=Path.cwd().resolve()
if not (PROJECT/'data').is_dir() and (PROJECT/'ventilator-pressure-prediction/data').is_dir():
    PROJECT=PROJECT/'ventilator-pressure-prediction'
ROOT=PROJECT/'outputs/improvements_v005'
SOURCES={'sequence':ROOT/'sequence','tree_context':ROOT/'tree_context'}
REVIEW=ROOT/'review'
VERIFY=os.environ.get('PA3_REVIEW_V005_VERIFY','1')=='1'
SAVE=os.environ.get('PA3_REVIEW_V005_SAVE','1')=='1'
SHOW_ONLINE=os.environ.get('PA3_REVIEW_V005_ONLINE','0')=='1'
EXPECTED_TRAIN_SHA='2dc30dc5c8e603c8c68200830eab7ed8c7af42c823f8ae68b0beb663eea817d7'
PRESELECTED_HARD_BREATHS=[59872,65844,18594,5758,114598,82930]
WARNINGS=[]
FIGURES=[]
TABLES=[]
print('Read-only v005 review at', datetime.now(timezone.utc).isoformat())
print('Project:',PROJECT,'| verification:',VERIFY,'| save review artifacts:',SAVE)


Read-only v005 review at 2026-11-04T16:23:33.663411+00:00
Project: /Users/generalclasher/Documents/dsc102/pa3/ventilator-pressure-prediction | verification: True | save review artifacts: True


In [2]:
def signature(value):
    return hashlib.sha256(json.dumps(value,sort_keys=True).encode()).hexdigest()

def sha(path):
    digest=hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda:stream.read(1024*1024),b''):
            digest.update(block)
    return digest.hexdigest()

def portable_path(value, fallback=None):
    if value:
        path=Path(value)
        if path.is_file():return path
        text=str(value).replace('\\','/')
        marker='/ventilator-pressure-prediction/'
        if marker in text:
            candidate=PROJECT/text.split(marker,1)[1]
            if candidate.is_file():return candidate
        if '/outputs/' in text:
            candidate=PROJECT/'outputs'/text.split('/outputs/',1)[1]
            if candidate.is_file():return candidate
    if fallback is not None and Path(fallback).is_file():return Path(fallback)
    return None

def read_json(path):
    return json.loads(Path(path).read_text())

def read_jsonl(path):
    path=Path(path)
    if not path.is_file():return []
    lines=path.read_text().splitlines()
    records=[]
    for index,line in enumerate(lines):
        if not line.strip():continue
        try:records.append(json.loads(line))
        except json.JSONDecodeError:
            # A live writer may be in the middle of its final line.
            WARNINGS.append(f'{path.name}: ignored malformed line {index+1}'+(' (live tail)' if index==len(lines)-1 else ''))
    return records

def confirmation_lock(directory):
    # Only this round's current root-level lock can authorize C summaries.
    # Archived locks, old B locks and lock-looking descendant names do not count.
    directory=Path(directory)
    for name in ['locked_before_C.json','locked_before_confirmation.json']:
        path=directory/name
        if not path.is_file():continue
        try:
            payload=read_json(path)
            if not isinstance(payload,dict) or not payload:continue
            design_path=directory/'design.json'
            if not design_path.is_file():continue
            if payload.get('lock_sha256')!=signature({key:value for key,value in payload.items() if key!='lock_sha256'}):continue
            if payload.get('design_sha256')!=signature(read_json(design_path)):continue
            if not isinstance(payload.get('candidates'),dict) or not payload['candidates']:continue
            approaches=payload.get('approaches',{})
            if set(approaches)!=set(['reference','best_individual','best_pair']):continue
            if payload.get('pair_weights_tuned_on_C') is not False:continue
            if payload.get('selected_using')!='A only; no C scores read':continue
            required=payload.get('C_required_members')
            if not isinstance(required,list) or not required or not set(required)<=set(payload['candidates']):continue
            if any(not isinstance(value,dict) or not value.get('members') or not set(value['members'])<=set(required) for value in approaches.values()):continue
            return path
        except (ValueError,OSError,TypeError,KeyError) as error:
            WARNINGS.append(f'Confirmation lock unreadable {path}: {error}')
    return None

def confirmation_summaries():
    rows=[]
    for directory in sorted(ROOT.glob('confirmation_round*')):
        if not directory.is_dir():continue
        lock=confirmation_lock(directory)
        if lock is None:
            rows.append(dict(round=directory.name,status='outcomes withheld: no confirmation lock',lock=None))
            continue
        for path in sorted(directory.rglob('*.json')):
            if any(part.lower() in ['archive','archives'] for part in path.relative_to(directory).parts):continue
            name=path.name.lower()
            if path==lock or not any(token in name for token in ['selection','manifest']):continue
            try:
                payload=read_json(path)
                if not isinstance(payload,dict):continue
                public={key:value for key,value in payload.items() if key in
                    ['approach','members','weights','eligible','A_mae','B_mae','C_mae',
                     'selected_name','selected_approach','selection_rule','final_refit_has_independent_holdout',
                     'training_breaths','prediction_rows','submission_sha256','csv_sha256']}
                rows.append(dict(round=directory.name,status='locked summary',lock=str(lock),file=str(path),**public))
            except (ValueError,OSError) as error:
                WARNINGS.append(f'Locked summary unreadable {path}: {error}')
    return rows

def discover_records():
    records=[]
    for source,directory in SOURCES.items():
        for path in sorted(directory.glob('record_A_*.json')):
            try:
                record=read_json(path)
                if not isinstance(record,dict):raise ValueError('Expected object')
                name=record.get('name',path.stem.removeprefix('record_A_'))
                report_path=portable_path(record.get('report_file'),directory/('report_A_'+name+'.json'))
                report=read_json(report_path) if report_path else None
                if source=='tree_context':
                    complete=record.get('status')=='complete'
                else:
                    cfg=record.get('recipe',{}).get('train_cfg',{})
                    complete=bool(report and report.get('completed') and
                        report.get('executed_epochs')==cfg.get('max_epochs') and
                        record.get('executed_epochs')==cfg.get('max_epochs'))
                records.append(dict(source=source,directory=directory,path=path,record=record,
                    report=report,report_path=report_path,name=name,complete=complete))
            except (ValueError,OSError,TypeError) as error:
                WARNINGS.append(f'Unusable record {path}: {error}')
    return records

def stage_progress(records):
    by_key={(item['source'],item['name']):item for item in records}
    rows=[];histories={}
    for path in sorted(SOURCES['sequence'].glob('A_*_epochs.jsonl')):
        name=path.name.removeprefix('A_').removesuffix('_epochs.jsonl')
        events=read_jsonl(path)
        epochs={int(event['epoch']):event for event in events if event.get('event')=='epoch' and 'epoch' in event}
        ordered=[epochs[key] for key in sorted(epochs)]
        histories[name]=ordered
        starts=[event for event in events if event.get('event') in ['start','resume']]
        cfg=starts[-1].get('train_cfg',{}) if starts else {}
        target=cfg.get('max_epochs',200)
        last=ordered[-1] if ordered else {}
        item=by_key.get(('sequence',name))
        complete=bool(item and item['complete'])
        recent=[event.get('epoch_seconds') for event in ordered[-10:] if event.get('epoch_seconds') is not None]
        estimate=(sum(recent)/len(recent))*max(0,target-last.get('epoch',0)) if recent else None
        rows.append(dict(source='sequence',name=name,status='complete' if complete else 'in progress; not ranked',
            phase=item['record'].get('phase') if item else None,
            completed_units=last.get('epoch',0),budget=target,unit='epochs',
            best_epoch=last.get('best_epoch'),best_so_far_A_mae=last.get('best_val_mae'),
            eval_mode_fit_sample_mae=last.get('training_eval_inspiratory_mae'),
            optimizer_updates=last.get('cumulative_optimizer_steps'),runtime_seconds=last.get('elapsed_seconds'),
            remaining_seconds_estimate=0. if complete else estimate,log_file=str(path)))
    tree_events=read_jsonl(SOURCES['tree_context']/'events.jsonl')
    tree_names={event['name'] for event in tree_events if event.get('event')=='trial_started' and 'name' in event}
    tree_names|={path.name.removeprefix('A_').removesuffix('_progress.jsonl') for path in SOURCES['tree_context'].glob('A_*_progress.jsonl')}
    for name in sorted(tree_names):
        path=SOURCES['tree_context']/('A_'+name+'_progress.jsonl')
        events=read_jsonl(path);last=events[-1] if events else {}
        item=by_key.get(('tree_context',name));complete=bool(item and item['complete'])
        record=item['record'] if item else {}
        rows.append(dict(source='tree_context',name=name,status='complete' if complete else 'in progress; not ranked',
            completed_units=2500 if complete else last.get('iteration',0),budget=2500,unit='trees',
            best_epoch=None,best_so_far_A_mae=record.get('validation_mae') if complete else None,
            runtime_seconds=record.get('runtime_seconds',last.get('elapsed_fit_seconds')),
            optimizer_updates=None,remaining_seconds_estimate=None,log_file=str(path)))
    # Completed imported records may legitimately have no live log (baseline).
    known={(row['source'],row['name']) for row in rows}
    for item in records:
        if (item['source'],item['name']) not in known:
            record=item['record'];tree=item['source']=='tree_context'
            rows.append(dict(source=item['source'],name=item['name'],status='complete' if item['complete'] else 'in progress; not ranked',
                completed_units=2500 if tree and item['complete'] else record.get('executed_epochs',0),
                budget=2500 if tree else record.get('recipe',{}).get('train_cfg',{}).get('max_epochs',200),
                unit='trees' if tree else 'epochs',best_epoch=record.get('best_epoch'),
                best_so_far_A_mae=record.get('validation_mae'),runtime_seconds=record.get('runtime_seconds',record.get('elapsed_seconds')),
                optimizer_updates=item.get('report',{}).get('optimizer_steps') if item.get('report') else None))
    return rows,histories

RECORDS=discover_records()
PROGRESS,HISTORIES=stage_progress(RECORDS)
CONFIRMATIONS=confirmation_summaries()
HAS_RESULTS=bool(RECORDS or PROGRESS or CONFIRMATIONS)
print('Discovered:',len(RECORDS),'A records;',sum(item['complete'] for item in RECORDS),'completed;',len(PROGRESS),'stage logs/records')
if not HAS_RESULTS:print('No v005 results are available locally yet. Rerun after artifacts are copied here.')


Discovered: 14 A records; 14 completed; 14 stage logs/records


## Completed records and unfinished budgets

Only records with a committed completed status/report enter the A ranking. Live best-so-far scores remain in the separate progress table and do not establish a final training duration. The remaining-time estimate uses recent complete epochs and is descriptive. Future C summaries are shown only when their round contains a confirmation lock; this review never opens C labels or predictions.


In [3]:
if HAS_RESULTS:
    import numpy as np
    import pandas as pd
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    from matplotlib.ticker import PercentFormatter
    try:
        from IPython.display import display
    except ImportError:
        def display(value):print(value.to_string(index=False) if hasattr(value,'to_string') else value)
    plt.rcParams.update({'figure.dpi':120,'font.size':10,'axes.spines.top':False,
        'axes.spines.right':False,'axes.grid':True,'grid.alpha':.2,'savefig.dpi':160})
    if SAVE:REVIEW.mkdir(parents=True,exist_ok=True)

def save_table(table,name):
    if SAVE and HAS_RESULTS:
        path=REVIEW/(name+'.csv');temporary=path.with_name(path.name+'.tmp')
        table.to_csv(temporary,index=False);os.replace(temporary,path);TABLES.append(str(path))
    return table

def finish_figure(figure,name):
    figure.tight_layout()
    if SAVE:
        for suffix in ['png','svg']:
            path=REVIEW/(name+'.'+suffix)
            figure.savefig(path,bbox_inches='tight');FIGURES.append(str(path))
    display(figure)
    plt.close(figure)

if HAS_RESULTS:
    progress_table=pd.DataFrame(PROGRESS)
    display(save_table(progress_table,'live_stage_progress'))
    completed_rows=[]
    for item in RECORDS:
        if not item['complete']:continue
        r=item['record'];report=item.get('report') or {};recipe=r.get('recipe',{})
        names=r.get('feature_names',r.get('input_columns',[]))
        completed_rows.append(dict(source=item['source'],name=item['name'],phase=r.get('phase','tree features'),
            feature_mode=r.get('feature_mode'),features=len(names) or r.get('n_features'),seed=r.get('seed'),
            config=json.dumps(recipe.get('model_spec',recipe.get('params',{})),sort_keys=True,separators=(',',':')),
            optimizer=recipe.get('train_cfg',{}).get('optimizer'),scheduler=recipe.get('train_cfg',{}).get('scheduler'),
            best_epoch=r.get('best_epoch'),executed_epochs=r.get('executed_epochs'),
            optimizer_updates=report.get('optimizer_steps'),trees=r.get('best_iteration'),
            runtime_seconds=r.get('runtime_seconds',r.get('elapsed_seconds')),
            eval_mode_fit_sample_mae=r.get('training_eval_mae'),A_mae=r.get('validation_mae'),
            reused_checkpoint=r.get('reused_checkpoint',False)))
    completed_table=pd.DataFrame(completed_rows)
    if len(completed_table):
        completed_table=completed_table.sort_values(['A_mae','source','name']).reset_index(drop=True)
        completed_table.insert(0,'completed_A_rank',np.arange(1,len(completed_table)+1))
        display(save_table(completed_table,'completed_A_records'))
    else:print('No committed complete records to rank.')
    if CONFIRMATIONS:display(pd.DataFrame(CONFIRMATIONS))


,source,name,status,phase,completed_units,budget,unit,best_epoch,best_so_far_A_mae,eval_mode_fit_sample_mae,optimizer_updates,runtime_seconds,remaining_seconds_estimate,log_file
0,sequence,capacity_bigru128,complete,capacity,200,200,epochs,191.0,0.262326,0.157546,47800.0,1466.036639,0.0,/Users/generalclasher/Documents/dsc102/pa3/ven...
1,sequence,capacity_bilstm128,complete,capacity,200,200,epochs,193.0,0.214166,0.139130,47800.0,2191.985584,0.0,/Users/generalclasher/Documents/dsc102/pa3/ven...
2,sequence,capacity_deep_bilstm,complete,capacity,200,200,epochs,189.0,0.211544,0.093103,47800.0,8478.753849,0.0,/Users/generalclasher/Documents/dsc102/pa3/ven...
3,sequence,capacity_uni_gru256,complete,capacity,200,200,epochs,198.0,0.268507,0.173757,47800.0,1664.286114,0.0,/Users/generalclasher/Documents/dsc102/pa3/ven...
4,sequence,features_combined,complete,features,200,200,epochs,193.0,0.334773,0.268096,47800.0,978.701827,0.0,/Users/generalclasher/Documents/dsc102/pa3/ven...
5,sequence,features_dynamics,complete,features,200,200,epochs,192.0,0.341036,0.285112,47800.0,961.516104,0.0,/Users/generalclasher/Documents/dsc102/pa3/ven...
6,sequence,features_offline,complete,features,200,200,epochs,192.0,0.351865,0.282674,47800.0,954.782548,0.0,/Users/generalclasher/Documents/dsc102/pa3/ven...
7,sequence,schedule_cosine_w5,complete,schedule,200,200,epochs,189.0,0.371982,0.312940,47800.0,913.999779,0.0,/Users/generalclasher/Documents/dsc102/pa3/ven...
8,sequence,schedule_plateau_p1,complete,schedule,200,200,epochs,200.0,0.565606,0.541505,47800.0,1039.679797,0.0,/Users/generalclasher/Documents/dsc102/pa3/ven...
9,sequence,schedule_plateau_p6,complete,schedule,200,200,epochs,193.0,0.415483,0.361931,47800.0,958.883658,0.0,/Users/generalclasher/Documents/dsc102/pa3/ven...


,completed_A_rank,source,name,phase,feature_mode,features,seed,config,optimizer,scheduler,best_epoch,executed_epochs,optimizer_updates,trees,runtime_seconds,eval_mode_fit_sample_mae,A_mae,reused_checkpoint
0,1,sequence,capacity_deep_bilstm,capacity,combined,47,20261102,"{""bidirectional"":true,""dropout"":0.15,""family"":...",Adam,replay,189.0,200.0,47800.0,NaN,8481.616000,0.094459,0.211544,False
1,2,sequence,capacity_bilstm128,capacity,combined,47,20261102,"{""bidirectional"":true,""dropout"":0.15,""family"":...",Adam,replay,193.0,200.0,47800.0,NaN,2192.449833,0.139321,0.214166,False
2,3,sequence,capacity_bigru128,capacity,combined,47,20261102,"{""bidirectional"":true,""dropout"":0.15,""family"":...",Adam,replay,191.0,200.0,47800.0,NaN,1466.325086,0.157217,0.262326,False
3,4,sequence,capacity_uni_gru256,capacity,combined,47,20261102,"{""bidirectional"":false,""dropout"":0.15,""family""...",Adam,replay,198.0,200.0,47800.0,NaN,1664.628687,0.172877,0.268507,False
4,5,sequence,features_combined,features,combined,47,20261102,"{""bidirectional"":false,""dropout"":0.15,""family""...",Adam,replay,193.0,200.0,47800.0,NaN,978.876878,0.266705,0.334773,False
5,6,sequence,features_dynamics,features,dynamics,33,20261102,"{""bidirectional"":false,""dropout"":0.15,""family""...",Adam,replay,192.0,200.0,47800.0,NaN,961.728484,0.284161,0.341036,False
6,7,sequence,features_offline,features,offline,34,20261102,"{""bidirectional"":false,""dropout"":0.15,""family""...",Adam,replay,192.0,200.0,47800.0,NaN,954.971790,0.281352,0.351865,False
7,8,sequence,schedule_cosine_w5,schedule,causal20,20,20261102,"{""bidirectional"":false,""dropout"":0.15,""family""...",Adam,cosine,189.0,200.0,47800.0,NaN,914.156698,0.313433,0.371982,False
8,9,sequence,schedule_plateau_p6,schedule,causal20,20,20261102,"{""bidirectional"":false,""dropout"":0.15,""family""...",Adam,plateau,193.0,200.0,47800.0,NaN,959.041609,0.361709,0.415483,False
9,10,tree_context,tree_combined,tree features,combined,47,20261102,"{""learning_rate"":0.04,""max_bin"":255,""min_child...",None,None,NaN,NaN,NaN,2500.0,469.247084,NaN,0.503760,False


,round,status,lock,file,training_breaths,members,weights,final_refit_has_independent_holdout,submission_sha256,approach,A_mae,C_mae,eligible
0,confirmation_round1,locked summary,/Users/generalclasher/Documents/dsc102/pa3/ven...,/Users/generalclasher/Documents/dsc102/pa3/ven...,67905.0,"[nn__capacity_bilstm128, nn__capacity_deep_bil...","[0.5, 0.5]",False,ff149f3ab87e1e52b948f536c9fdf9a526471ac0723b6b...,NaN,NaN,NaN,NaN
1,confirmation_round1,locked summary,/Users/generalclasher/Documents/dsc102/pa3/ven...,/Users/generalclasher/Documents/dsc102/pa3/ven...,NaN,"[nn__capacity_bilstm128, nn__capacity_deep_bil...","[0.5, 0.5]",False,NaN,best_pair,0.196563,0.210612,True


## Native-unit learning curves

Blue is held-out A inspiration MAE; orange is a fixed fitting-breath sample scored in evaluation mode. The two use different breath samples. The optional gray online curve also includes training-mode stochastic behavior and changing weights. Dashed epoch markers identify completed records' restored checkpoints; unfinished histories are labeled in progress.


In [4]:
if HAS_RESULTS and HISTORIES:
    names=sorted(HISTORIES)
    columns=min(3,len(names));rows=math.ceil(len(names)/columns)
    figure,axes=plt.subplots(rows,columns,figsize=(5.1*columns,3.5*rows),squeeze=False)
    completed={item['name']:item for item in RECORDS if item['source']=='sequence' and item['complete']}
    for axis,name in zip(axes.ravel(),names):
        history=HISTORIES[name]
        if not history:
            axis.set_title(name+' — awaiting first epoch');continue
        epoch=[event['epoch'] for event in history]
        axis.plot(epoch,[event.get('validation_inspiratory_mae',np.nan) for event in history],label='A validation',color='#2563eb')
        axis.plot(epoch,[event.get('training_eval_inspiratory_mae',np.nan) for event in history],label='Fit sample, eval mode',color='#ea8b23')
        if SHOW_ONLINE:
            axis.plot(epoch,[event.get('training_online_inspiratory_mae',np.nan) for event in history],label='Online train mode',color='#888888',linestyle=':',alpha=.8)
        if name in completed:
            selected=completed[name]['record'].get('best_epoch')
            if selected is not None:axis.axvline(selected,color='#334155',linestyle='--',linewidth=1,label='Restored best epoch')
        state='complete' if name in completed else 'IN PROGRESS'
        axis.set_title(name+'\n'+state,fontsize=10)
        axis.set(xlabel='Executed epoch',ylabel='Inspiratory MAE (native pressure units)')
        axis.legend(fontsize=8)
    for axis in axes.ravel()[len(names):]:axis.set_visible(False)
    finish_figure(figure,'native_learning_curves')
elif HAS_RESULTS:print('No sequence epoch histories available yet.')


<Figure size 1836x1680 with 12 Axes>

## Portable, read-only artifact checks

Sequence prediction hashes cover array bytes; tree prediction hashes cover the `.npy` file. Paths saved on DataHub map back to the same repository-relative path when viewed locally. Checks read serialized files without loading model code or executing checkpoints. Missing files are reported separately from mismatches. Counts are derived from discovered records rather than a fixed expected model count.


In [5]:
def source_code_sha(path):
    notebook=read_json(path)
    code='\n'.join(''.join(cell['source']) for cell in notebook['cells'] if cell['cell_type']=='code')
    return hashlib.sha256(code.encode()).hexdigest()

def expected_feature_names(mode):
    path=PROJECT/'research/feature_components_v005.ipynb'
    if not path.is_file():raise FileNotFoundError('Feature-schema source unavailable')
    values={}
    for cell in read_json(path)['cells']:
        if 'v005-feature-definition' not in cell.get('metadata',{}).get('tags',[]):continue
        for node in ast.parse(''.join(cell['source'])).body:
            if isinstance(node,ast.Assign):
                for target in node.targets:
                    if isinstance(target,ast.Name) and target.id in ['V005_BASE_FEATURE_NAMES','V005_OFFLINE_FEATURE_NAMES','V005_DYNAMICS_FEATURE_NAMES']:
                        values[target.id]=list(ast.literal_eval(node.value))
    names=values['V005_BASE_FEATURE_NAMES']
    if mode in ['offline','combined']:names+=values['V005_OFFLINE_FEATURE_NAMES']
    if mode in ['dynamics','combined']:names+=values['V005_DYNAMICS_FEATURE_NAMES']
    return names

def row_id_path(item):
    r=item['record'];directory=item['directory']
    return portable_path(r.get('scored_row_id_file'),directory/'A_validation_row_ids.npy')

def load_prediction(item):
    r=item['record'];path=portable_path(r.get('prediction_file'),item['directory']/('A_'+item['name']+'_predictions.npy'))
    if path is None:raise FileNotFoundError('Prediction artifact unavailable')
    prediction=np.load(path,allow_pickle=False)
    if prediction.ndim!=1 or not np.isfinite(prediction).all():raise ValueError('Invalid prediction vector')
    expected=r.get('prediction_sha256')
    actual=sha(path) if item['source']=='tree_context' else hashlib.sha256(np.ascontiguousarray(prediction).tobytes()).hexdigest()
    if expected is None or actual!=expected:raise ValueError('Prediction hash mismatch')
    rowpath=row_id_path(item)
    if rowpath is None:raise FileNotFoundError('Scored row-ID artifact unavailable')
    rowids=np.load(rowpath,allow_pickle=False)
    if rowids.shape!=prediction.shape or len(np.unique(rowids))!=len(rowids):raise ValueError('Row-ID shape/uniqueness mismatch')
    if r.get('scored_row_id_sha256') and sha(rowpath)!=r['scored_row_id_sha256']:raise ValueError('Row-ID file hash mismatch')
    original=PROJECT/'outputs/controlled_v003/A_validation_row_ids.npy'
    if not original.is_file():raise FileNotFoundError('Original A row-ID anchor unavailable')
    if not np.array_equal(rowids,np.load(original,allow_pickle=False)):raise ValueError('Original A row order mismatch')
    return prediction.astype(np.float64,copy=False),rowids

def verify_record(item):
    r=item['record'];checks={};missing=[];errors=[]
    def run(label,function):
        try:
            value=function()
            if value is False:raise ValueError('check returned false')
            checks[label]='passed'
        except FileNotFoundError as error:checks[label]='missing';missing.append(label+': '+str(error))
        except (ValueError,KeyError,AssertionError,OSError,TypeError) as error:checks[label]='failed';errors.append(label+': '+str(error))
    design_path=item['directory']/'design.json'
    design=read_json(design_path) if design_path.is_file() else {}
    def design_identity():
        if not design:raise FileNotFoundError('Design unavailable')
        if item['source']=='sequence':
            assert r['identity']==signature(dict(recipe=r['recipe'],design=signature(design),split='A'))
            assert r['fit_group_sha256']==design['group_hashes']['A_fit']
        else:
            assert r['identity_sha256']==signature(r['identity'])
            assert r['identity']['design_sha256']==signature(design)
            assert r['identity']['fit_breath_sha256']==design['group_hashes']['A_fit']
        assert design['train_sha256']==EXPECTED_TRAIN_SHA
    run('design_and_record_identity',design_identity)
    def source_identity():
        path=PROJECT/('iterative_improvements_v005.ipynb' if item['source']=='sequence' else 'tree_context_v005.ipynb')
        if not path.is_file():raise FileNotFoundError('Source notebook unavailable')
        expected=design.get('source_code_sha256',design.get('source_sha256'))
        if not expected:raise FileNotFoundError('Design source hash unavailable')
        assert source_code_sha(path)==expected
    run('full_source_code_hash',source_identity)
    def split_identity():
        path=item['directory']/'split_breath_ids.npz'
        if not path.is_file():raise FileNotFoundError('Saved groups unavailable')
        with np.load(path,allow_pickle=False) as saved:
            fit=saved['A_fit'];val=saved['A_validation']
        assert len(fit)==61114 and len(val)==6791 and len(np.unique(fit))==len(fit) and len(np.unique(val))==len(val)
        assert not np.intersect1d(fit,val).size
        for key,values in [('A_fit',fit),('A_validation',val)]:
            assert hashlib.sha256(np.ascontiguousarray(values).tobytes()).hexdigest()==design['group_hashes'][key]
        original=PROJECT/'outputs/controlled_v003/split_breath_ids.npz'
        if not original.is_file():raise FileNotFoundError('Original A split anchor unavailable')
        with np.load(original,allow_pickle=False) as saved:
            assert np.array_equal(fit,saved['A_fit']) and np.array_equal(val,saved['A_validation'])
    run('original_group_hashes_and_exclusion',split_identity)
    def features():
        names=r.get('feature_names',r.get('input_columns',[]));mode=r['feature_mode']
        if item['source']=='sequence':assert len(names)==design['feature_mode_dimensions'][mode]
        else:assert names==design['feature_names'][mode]
        assert names==expected_feature_names(mode)
        if item.get('report') and item['report'].get('preprocessing'):
            assert names==item['report']['preprocessing']['feature_names']
        assert len(set(names))==len(names) and not set(names)&{'id','breath_id','pressure'}
    run('feature_order_dimensions_no_target_or_ID',features)
    def checkpoint():
        path=portable_path(r.get('checkpoint_path',r.get('model_file')))
        if path is None:raise FileNotFoundError('Checkpoint unavailable')
        assert sha(path)==r.get('checkpoint_sha256',r.get('model_sha256'))
    run('checkpoint_file_hash',checkpoint)
    run('prediction_hash_and_original_scored_ID_alignment',lambda:load_prediction(item) is not None)
    if item['source']=='sequence':
        run('full_budget_completed_report',lambda:bool(item['report'] and item['report']['completed'] and
            item['report']['executed_epochs']==r['recipe']['train_cfg']['max_epochs']==r['executed_epochs']))
    else:
        run('fixed_tree_budget_complete',lambda:r['status']=='complete' and r['best_iteration']==2500 and r['validation_feedback'] is False)
    return dict(source=item['source'],name=item['name'],status='failed' if errors else 'available checks passed; missing artifacts' if missing else 'passed',
        passed=sum(value=='passed' for value in checks.values()),missing=len(missing),failed=len(errors),checks=checks,missing_details=missing,error_details=errors)

ARTIFACT_CHECKS=[]
if HAS_RESULTS and VERIFY:
    for item in RECORDS:
        if item['complete']:ARTIFACT_CHECKS.append(verify_record(item))
    if ARTIFACT_CHECKS:
        display(pd.DataFrame([{key:value for key,value in row.items() if key in ['source','name','status','passed','missing','failed']} for row in ARTIFACT_CHECKS]))
else:
    if HAS_RESULTS:print('Artifact verification skipped by PA3_REVIEW_V005_VERIFY=0.')


,source,name,status,passed,missing,failed
0,sequence,capacity_bigru128,passed,7,0,0
1,sequence,capacity_bilstm128,passed,7,0,0
2,sequence,capacity_deep_bilstm,passed,7,0,0
3,sequence,capacity_uni_gru256,passed,7,0,0
4,sequence,features_combined,passed,7,0,0
5,sequence,features_dynamics,passed,7,0,0
6,sequence,features_offline,passed,7,0,0
7,sequence,schedule_cosine_w5,passed,7,0,0
8,sequence,schedule_plateau_p1,passed,7,0,0
9,sequence,schedule_plateau_p6,passed,7,0,0


## A error contribution and frozen hard examples

The diagnostic plots use the lowest-MAE **completed** sequence model and tree with available aligned predictions, compared with the previously frozen v004b validation blend. This review creates no new selection or ensemble weight. Every prediction is checked against original scored A row IDs and approved raw labels.

Error share measures total absolute error, so large subgroups can matter even without the largest MAE. Inspiration phases divide each breath's scored positions into early/middle/late thirds, matching v004. The six hard breath IDs were selected in v004 from the old GRU's largest total errors and remain fixed here; they are selected cases, not representative examples. Analysis waits until at least one fresh v005 model is complete.


In [6]:
def approved_A_rows(expected_ids):
    path=PROJECT/'data/train.csv'
    if not path.is_file():raise FileNotFoundError('Approved training CSV unavailable')
    if sha(path)!=EXPECTED_TRAIN_SHA:raise ValueError('Approved training CSV hash mismatch')
    wanted=set(np.asarray(expected_ids).tolist());chunks=[]
    dtypes=dict(id='int64',breath_id='int32',R='int8',C='int8',time_step='float32',u_out='int8',pressure='float32')
    for chunk in pd.read_csv(path,usecols=list(dtypes),dtype=dtypes,chunksize=400000):
        retained=chunk[chunk.id.isin(wanted)]
        if len(retained):chunks.append(retained)
    rows=pd.concat(chunks,ignore_index=True) if chunks else pd.DataFrame()
    if not len(rows) or not np.array_equal(rows.id.to_numpy(),expected_ids):raise ValueError('Raw-label row order mismatch')
    assert (rows.u_out==0).all() and rows.id.is_unique and rows.breath_id.nunique()==6791
    rows['absolute_step']=rows.groupby('breath_id',sort=False).cumcount().to_numpy()
    count=rows.groupby('breath_id',sort=False).id.transform('size').to_numpy()
    fraction=rows.absolute_step.to_numpy()/np.maximum(count-1,1)
    rows['inspiration_phase']=np.select([fraction<1/3,fraction<2/3],['early','middle'],default='late')
    return rows

def prior_v004_blend(expected_ids):
    directory=PROJECT/'outputs/sequence_v004b/experiments'
    lock_path=directory/'locked_before_B.json'
    if not lock_path.is_file():raise FileNotFoundError('Prior frozen blend lock unavailable')
    lock=read_json(lock_path)
    assert lock['pair_members']==['tree_511','warm_current'] and lock['pair_weights']==[.25,.75]
    predictions=[]
    for name in lock['pair_members']:
        path=directory/('record_A_'+name+'.json')
        if not path.is_file():raise FileNotFoundError('Prior A member record unavailable: '+name)
        record=read_json(path)
        prediction_path=portable_path(record.get('prediction_file'),directory/('A_'+name+'_predictions.npy'))
        if prediction_path is None:raise FileNotFoundError('Prior A member predictions unavailable: '+name)
        value=np.load(prediction_path,allow_pickle=False)
        assert hashlib.sha256(np.ascontiguousarray(value).tobytes()).hexdigest()==record['prediction_sha256']
        assert value.shape==expected_ids.shape and np.isfinite(value).all()
        predictions.append(value.astype(np.float64,copy=False))
    rowpath=directory/'A_row_ids.npy'
    if not rowpath.is_file():raise FileNotFoundError('Prior A row-ID artifact unavailable')
    assert np.array_equal(np.load(rowpath,allow_pickle=False),expected_ids)
    return sum(weight*value for weight,value in zip(lock['pair_weights'],predictions)),lock['pair_A_mae']

def contribution_table(rows,predictions,columns):
    tables=[];target=rows.pressure.to_numpy(dtype=np.float64)
    for name,prediction in predictions.items():
        frame=rows[columns].copy();frame['absolute_error']=np.abs(prediction-target)
        table=frame.groupby(columns,observed=True).absolute_error.agg(['mean','size','sum']).reset_index()
        table=table.rename(columns={'mean':'mae','size':'scored_points','sum':'total_absolute_error'})
        table.insert(0,'model',name)
        table['point_share']=table.scored_points/len(target)
        table['total_error_share']=table.total_absolute_error/table.total_absolute_error.sum()
        assert int(table.scored_points.sum())==len(target)
        assert np.isclose(table.total_absolute_error.sum(),np.abs(prediction-target).sum())
        tables.append(table)
    return pd.concat(tables,ignore_index=True)

ANALYSIS_MODELS={};A_ROWS=None;ANALYSIS_STATUS='deferred: no fresh completed A model'
if HAS_RESULTS and any(item['complete'] and not item['record'].get('reused_checkpoint',False) for item in RECORDS):
    try:
        anchor=PROJECT/'outputs/controlled_v003/A_validation_row_ids.npy'
        if not anchor.is_file():raise FileNotFoundError('Original A scored-row anchor unavailable')
        expected_ids=np.load(anchor,allow_pickle=False)
        # Try candidates in completed order; missing artifacts never substitute a partial stage.
        for source,label in [('sequence','v005 sequence'),('tree_context','v005 tree')]:
            candidates=sorted([item for item in RECORDS if item['source']==source and item['complete']],key=lambda item:(item['record']['validation_mae'],item['name']))
            for item in candidates:
                try:
                    prediction,rowids=load_prediction(item)
                    ANALYSIS_MODELS[label+': '+item['name']]=prediction
                    break
                except (ValueError,OSError,AssertionError) as error:
                    WARNINGS.append('Analysis candidate skipped '+item['name']+': '+str(error))
        if not ANALYSIS_MODELS:raise FileNotFoundError('No complete aligned v005 prediction artifacts available')
        A_ROWS=approved_A_rows(expected_ids)
        target=A_ROWS.pressure.to_numpy(dtype=np.float64)
        for label,prediction in ANALYSIS_MODELS.items():
            item=next(item for item in RECORDS if label.endswith(': '+item['name']))
            assert np.isclose(np.abs(prediction-target).mean(),item['record']['validation_mae'],rtol=0,atol=1e-10)
        try:
            prior,prior_mae=prior_v004_blend(expected_ids)
            assert np.isclose(np.abs(prior-target).mean(),prior_mae,rtol=0,atol=1e-10)
            ANALYSIS_MODELS={'v004b frozen validation blend':prior,**ANALYSIS_MODELS}
        except (FileNotFoundError,ValueError,AssertionError) as error:
            WARNINGS.append('Prior blend comparison deferred: '+str(error))
        for partition,columns in [('R',['R']),('C',['C']),('R_C',['R','C']),('absolute_step',['absolute_step']),('inspiration_phase',['inspiration_phase'])]:
            table=contribution_table(A_ROWS,ANALYSIS_MODELS,columns)
            save_table(table,'A_error_by_'+partition)
            if partition in ['R_C','inspiration_phase']:display(table)
        ANALYSIS_STATUS='complete: approved labels and scored IDs verified; diagnostics only'
    except (ValueError,OSError,AssertionError,KeyError) as error:
        A_ROWS=None;ANALYSIS_MODELS={};ANALYSIS_STATUS='deferred: '+str(error)
print('A error analysis:',ANALYSIS_STATUS)

def verify_all_saved_MAE(records,artifact_checks,rows):
    if rows is None:return
    target=rows.pressure.to_numpy(dtype=np.float64)
    by_key={(item['source'],item['name']):item for item in records}
    for result in artifact_checks:
        item=by_key[(result['source'],result['name'])]
        label='native_MAE_from_approved_A_labels'
        try:
            prediction,rowids=load_prediction(item)
            assert np.array_equal(rowids,rows.id.to_numpy())
            mae=float(np.abs(prediction-target).mean())
            assert np.isclose(mae,item['record']['validation_mae'],rtol=0,atol=1e-10)
            result['checks'][label]='passed';result['passed']+=1
            result['recomputed_A_mae']=mae
        except FileNotFoundError as error:
            result['checks'][label]='missing';result['missing']+=1
            result['missing_details'].append(label+': '+str(error))
        except (ValueError,OSError,AssertionError) as error:
            result['checks'][label]='failed';result['failed']+=1
            result['error_details'].append(label+': '+str(error))
        result['status']='failed' if result['failed'] else 'available checks passed; missing artifacts' if result['missing'] else 'passed'

if HAS_RESULTS and VERIFY and A_ROWS is not None:
    verify_all_saved_MAE(RECORDS,ARTIFACT_CHECKS,A_ROWS)
    display(pd.DataFrame([{key:value for key,value in item.items() if key in ['source','name','status','passed','missing','failed','recomputed_A_mae']} for item in ARTIFACT_CHECKS]))


A error analysis: complete: approved labels and scored IDs verified; diagnostics only


,model,R,C,mae,scored_points,total_absolute_error,point_share,total_error_share
0,v004b frozen validation blend,5,10,0.449319,22461,10092.145557,0.108982,0.101232
1,v004b frozen validation blend,5,20,0.345984,23070,7981.843126,0.111937,0.080064
2,v004b frozen validation blend,5,50,0.254922,22061,5623.842090,0.107041,0.056411
3,v004b frozen validation blend,20,10,0.473874,16589,7861.095785,0.080491,0.078853
4,v004b frozen validation blend,20,20,0.365062,16704,6098.001278,0.081049,0.061167
5,v004b frozen validation blend,20,50,0.368712,21893,8072.214141,0.106226,0.080970
6,v004b frozen validation blend,50,10,0.654117,37444,24492.749633,0.181681,0.245680
7,v004b frozen validation blend,50,20,0.619849,23137,14341.454815,0.112262,0.143855
8,v004b frozen validation blend,50,50,0.665389,22739,15130.269433,0.110331,0.151768
9,v005 sequence: capacity_deep_bilstm,5,10,0.199121,22461,4472.457423,0.108982,0.102582


,model,inspiration_phase,mae,scored_points,total_absolute_error,point_share,total_error_share
0,v004b frozen validation blend,early,0.410172,69378,28456.917305,0.336626,0.285444
1,v004b frozen validation blend,late,0.527514,70938,37420.754084,0.344195,0.375358
2,v004b frozen validation blend,middle,0.514061,65782,33815.944468,0.319178,0.339199
3,v005 sequence: capacity_deep_bilstm,early,0.180220,69378,12503.325018,0.336626,0.286781
4,v005 sequence: capacity_deep_bilstm,late,0.224014,70938,15891.106184,0.344195,0.364485
5,v005 sequence: capacity_deep_bilstm,middle,0.231133,65782,15204.384167,0.319178,0.348734
6,v005 tree: tree_combined,early,0.362924,69378,25178.911500,0.336626,0.242515
7,v005 tree: tree_combined,late,0.582456,70938,41318.263366,0.344195,0.397965
8,v005 tree: tree_combined,middle,0.567431,65782,37326.766816,0.319178,0.359520


,source,name,status,passed,missing,failed,recomputed_A_mae
0,sequence,capacity_bigru128,passed,8,0,0,0.262326
1,sequence,capacity_bilstm128,passed,8,0,0,0.214166
2,sequence,capacity_deep_bilstm,passed,8,0,0,0.211544
3,sequence,capacity_uni_gru256,passed,8,0,0,0.268507
4,sequence,features_combined,passed,8,0,0,0.334773
5,sequence,features_dynamics,passed,8,0,0,0.341036
6,sequence,features_offline,passed,8,0,0,0.351865
7,sequence,schedule_cosine_w5,passed,8,0,0,0.371982
8,sequence,schedule_plateau_p1,passed,8,0,0,0.565606
9,sequence,schedule_plateau_p6,passed,8,0,0,0.415483


In [7]:
if A_ROWS is not None and ANALYSIS_MODELS:
    figure,axes=plt.subplots(2,2,figsize=(13,7.4))
    colors=['#64748b','#2563eb','#e58921']
    labels=list(ANALYSIS_MODELS)
    for column_index,column in enumerate(['R','C']):
        table=contribution_table(A_ROWS,ANALYSIS_MODELS,[column]);levels=sorted(A_ROWS[column].unique())
        width=.8/len(labels)
        for index,label in enumerate(labels):
            part=table[table.model==label].set_index(column).reindex(levels)
            x=np.arange(len(levels))+(index-(len(labels)-1)/2)*width
            for axis,metric in [(axes[0,column_index],'total_error_share'),(axes[1,column_index],'mae')]:
                axis.bar(x,part[metric],width=width,label=label,color=colors[index%len(colors)])
                axis.set_xticks(np.arange(len(levels)),[str(level) for level in levels]);axis.set_xlabel(column)
        axes[0,column_index].set_ylabel('Share of total absolute error');axes[0,column_index].yaxis.set_major_formatter(PercentFormatter(1))
        axes[1,column_index].set_ylabel('Inspiratory MAE (native units)')
        axes[0,column_index].set_title(column+' error contribution')
    axes[0,0].legend(fontsize=8)
    finish_figure(figure,'A_R_C_error_contributions')
    figure,axes=plt.subplots(1,2,figsize=(13,4.2))
    table=contribution_table(A_ROWS,ANALYSIS_MODELS,['absolute_step'])
    for index,label in enumerate(labels):
        part=table[table.model==label]
        axes[0].plot(part.absolute_step,part.mae,label=label,color=colors[index%len(colors)])
        axes[1].plot(part.absolute_step,part.total_error_share,label=label,color=colors[index%len(colors)])
    axes[0].set(xlabel='Absolute breath step (inspiration rows only)',ylabel='MAE (native units)')
    axes[1].set(xlabel='Absolute breath step (inspiration rows only)',ylabel='Share of total absolute error')
    axes[1].yaxis.set_major_formatter(PercentFormatter(1));axes[0].legend(fontsize=8)
    finish_figure(figure,'A_step_error_profile')
    figure,axis=plt.subplots(figsize=(10,4.2));table=contribution_table(A_ROWS,ANALYSIS_MODELS,['inspiration_phase']);phases=['early','middle','late'];width=.8/len(labels)
    for index,label in enumerate(labels):
        part=table[table.model==label].set_index('inspiration_phase').reindex(phases)
        axis.bar(np.arange(3)+(index-(len(labels)-1)/2)*width,part.total_error_share,width,label=label,color=colors[index%len(colors)])
    axis.set_xticks(np.arange(3),phases);axis.set_ylabel('Share of total absolute error');axis.yaxis.set_major_formatter(PercentFormatter(1));axis.legend(fontsize=8)
    finish_figure(figure,'A_inspiration_phase_contributions')


<Figure size 1560x888 with 4 Axes>

<Figure size 1560x504 with 2 Axes>

<Figure size 1200x504 with 1 Axes>

In [8]:
if A_ROWS is not None and ANALYSIS_MODELS:
    figure,axes=plt.subplots(2,3,figsize=(15,8),squeeze=False)
    target=A_ROWS.pressure.to_numpy(dtype=np.float64);hard_rows=[]
    for axis,breath in zip(axes.ravel(),PRESELECTED_HARD_BREATHS):
        mask=A_ROWS.breath_id.to_numpy()==breath
        if not mask.any():raise ValueError('Preselected hard breath missing from A')
        part=A_ROWS.loc[mask];axis.plot(part.time_step,target[mask],color='#111827',linewidth=2.2,label='True pressure')
        for index,(label,prediction) in enumerate(ANALYSIS_MODELS.items()):
            axis.plot(part.time_step,prediction[mask],color=colors[index%len(colors)],label=label,linewidth=1.5)
            error=np.abs(prediction[mask]-target[mask])
            hard_rows.append(dict(breath_id=breath,R=int(part.R.iloc[0]),C=int(part.C.iloc[0]),model=label,scored_points=int(mask.sum()),mae=float(error.mean()),total_absolute_error=float(error.sum())))
        axis.set_title(f'Breath {breath} | R={int(part.R.iloc[0])}, C={int(part.C.iloc[0])}')
        axis.set(xlabel='Time (s), inspiration only',ylabel='Pressure (native units)')
    axes[0,0].legend(fontsize=7)
    figure.suptitle('Six frozen v004 hard cases — selected examples, not representative breaths',fontsize=13)
    finish_figure(figure,'preselected_hard_breaths')
    save_table(pd.DataFrame(hard_rows),'preselected_hard_breath_comparison')


<Figure size 1800x960 with 6 Axes>

## Review provenance

A/C validation evidence belongs to held-out validation models. Any final all-labeled manifest describes separately fitted models and cannot provide a new holdout MAE. The review reports observed artifacts and diagnostic plots; model or ensemble adoption remains in the locked experiment workflow.


In [9]:
REVIEW_SUMMARY=dict(created_at_utc=datetime.now(timezone.utc).isoformat(),read_only_models=True,
    discovered_A_records=len(RECORDS),completed_A_records=sum(item['complete'] for item in RECORDS),
    in_progress_stages=sum(row['status']!='complete' for row in PROGRESS),
    record_counts_by_source={source:sum(item['source']==source for item in RECORDS) for source in SOURCES},
    artifact_records_checked=len(ARTIFACT_CHECKS),
    artifact_checks_passed=sum(item['passed'] for item in ARTIFACT_CHECKS),
    artifact_checks_missing=sum(item['missing'] for item in ARTIFACT_CHECKS),
    artifact_checks_failed=sum(item['failed'] for item in ARTIFACT_CHECKS),
    artifact_checks=ARTIFACT_CHECKS,confirmation_summaries=CONFIRMATIONS,
    error_analysis_status=ANALYSIS_STATUS,figures=FIGURES,tables=TABLES,warnings=WARNINGS,
    training_or_submission_performed=False)
if HAS_RESULTS and SAVE:
    path=REVIEW/'review_manifest.json';temporary=path.with_name(path.name+'.tmp')
    temporary.write_text(json.dumps(REVIEW_SUMMARY,indent=2,allow_nan=False)+'\n');os.replace(temporary,path)
print('Review:',REVIEW_SUMMARY['completed_A_records'],'completed A records;',REVIEW_SUMMARY['in_progress_stages'],'in-progress stages;',REVIEW_SUMMARY['artifact_checks_failed'],'failed artifact checks')
for warning in WARNINGS:print('REVIEW NOTE:',warning)


Review: 14 completed A records; 0 in-progress stages; 0 failed artifact checks


## Verified all-data submission and approved follow-up

The equal LSTM ensemble was freshly fitted on all 67,905 approved labeled breaths
for 193 / 189 fixed epochs. Its 603,600-row CSV passed approved-file hashes,
model/report hashes, training-only scaler count and target-statistic checks, exact
test/sample ID alignment, and independent CPU inference on 32 test breaths.
AI Coding Gym accepted it at **official MAE 0.18550**, compared with interim
0.19486. A validation MAE was 0.196563 and grouped C was 0.210612 for earlier
validation models; these scores are not holdouts for the final all-data models.
C is resampling confirmation, not an untouched test.

Evidence: `research/final_validation_v005.json`,
`research/final_inference_check_v005.json`,
`research/submission_receipt_v005_final.json`.

Approved H1/H2 partial results received before mandatory SSH/Duo renewal:
extra 100 epochs: compact A MAE 0.213908 (baseline 0.214166); slower 300-epoch
cosine: 0.206321. R/C-specific heads and fixed equal-pair comparisons need
server retrieval. No C decision is made from incomplete results.
